In [1]:
from pathlib import Path
import pandas as pd

# Project root
PROJECT_ROOT = Path.cwd().parent

# VisA dataset
VISA_PATH = PROJECT_ROOT / "datasets" / "VisA_20220922"

# Read candle annotations
candle_annotation = VISA_PATH / "candle" / "image_anno.csv"

df = pd.read_csv(candle_annotation)

print("Number of rows:", len(df))
print("\nColumns:")
print(df.columns.tolist())

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nFirst defective samples:")
print(df[df["label"] != "normal"].head(10).to_string(index=False))

Number of rows: 1100

Columns:
['image', 'label', 'mask']

Label distribution:
label
normal                                                                    1000
damaged corner of packaging                                                 18
different colour spot                                                       15
chunk of wax missing                                                        12
foreign particals on candle                                                 12
wax melded out of the candle                                                12
weird candle wick                                                            9
extra wax in candle                                                          9
damaged corner of packaging,foreign particals on candle                      2
chunk of wax missing,foreign particals on candle                             1
chunk of wax missing,different colour spot                                   1
damaged corner of packaging,different colour s

In [2]:
# Find all VisA categories

categories = sorted([
    folder.name
    for folder in VISA_PATH.iterdir()
    if folder.is_dir() and folder.name != "split_csv"
])

print("Number of categories:", len(categories))
print("\nCategories:")

for category in categories:
    print("-", category)

Number of categories: 12

Categories:
- candle
- capsules
- cashew
- chewinggum
- fryum
- macaroni1
- macaroni2
- pcb1
- pcb2
- pcb3
- pcb4
- pipe_fryum


In [3]:
# Analyze annotations for all VisA categories

all_visa_data = []

for category in categories:
    annotation_file = VISA_PATH / category / "image_anno.csv"

    if not annotation_file.exists():
        print(f"WARNING: Annotation file not found for {category}")
        continue

    df_category = pd.read_csv(annotation_file)

    df_category["category"] = category

    all_visa_data.append(df_category)

visa_annotations = pd.concat(
    all_visa_data,
    ignore_index=True
)

print("Total annotated images:", len(visa_annotations))

print("\nCategories:")
print(visa_annotations["category"].nunique())

print("\nOverall label distribution:")
print(visa_annotations["label"].eq("normal").value_counts())

print("\nImages per category:")
print(
    visa_annotations
    .groupby("category")
    .size()
    .sort_index()
)

Total annotated images: 10821

Categories:
12

Overall label distribution:
label
True     9621
False    1200
Name: count, dtype: int64

Images per category:
category
candle        1100
capsules       702
cashew         600
chewinggum     603
fryum          600
macaroni1     1100
macaroni2     1100
pcb1          1104
pcb2          1101
pcb3          1106
pcb4          1105
pipe_fryum     600
dtype: int64


In [4]:
# Normal vs anomalous images for every VisA category

visa_annotations["binary_label"] = (
    visa_annotations["label"].ne("normal").astype(int)
)

summary = (
    visa_annotations
    .groupby(["category", "binary_label"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={
        0: "normal",
        1: "anomaly"
    })
)

print(summary)

binary_label  normal  anomaly
category                     
candle          1000      100
capsules         602      100
cashew           500      100
chewinggum       503      100
fryum            500      100
macaroni1       1000      100
macaroni2       1000      100
pcb1            1004      100
pcb2            1001      100
pcb3            1006      100
pcb4            1005      100
pipe_fryum       500      100


In [5]:
# Check VisA mask paths

anomaly_data = visa_annotations[
    visa_annotations["label"] != "normal"
].copy()

def check_mask_exists(row):
    mask_value = str(row["mask"]).strip()

    if not mask_value:
        return False

    mask_path = VISA_PATH / mask_value
    return mask_path.exists()


anomaly_data["mask_exists"] = anomaly_data.apply(
    check_mask_exists,
    axis=1
)

print("Total anomalous images:", len(anomaly_data))
print("Masks found:", anomaly_data["mask_exists"].sum())
print("Masks missing:", (~anomaly_data["mask_exists"]).sum())

Total anomalous images: 1200
Masks found: 1200
Masks missing: 0


In [6]:
# Check VisA image paths

def check_image_exists(row):
    image_path = VISA_PATH / row["image"]
    return image_path.exists()


visa_annotations["image_exists"] = visa_annotations.apply(
    check_image_exists,
    axis=1
)

print("Total images:", len(visa_annotations))
print("Images found:", visa_annotations["image_exists"].sum())
print("Images missing:", (~visa_annotations["image_exists"]).sum())

Total images: 10821
Images found: 10821
Images missing: 0


In [7]:
# Create standardized VisA metadata

visa_metadata = visa_annotations.copy()

# Convert original VisA labels to our project-level binary label
visa_metadata["label"] = (
    visa_metadata["label"] != "normal"
).astype(int)

# Preserve the original annotation
visa_metadata["defect_type"] = visa_annotations["label"]

# Convert relative image paths to full project paths
visa_metadata["image_path"] = visa_annotations["image"].apply(
    lambda x: str(VISA_PATH / x)
)

# Convert mask paths
visa_metadata["mask_path"] = visa_annotations["mask"].apply(
    lambda x: "" if pd.isna(x) or str(x).strip() == ""
    else str(VISA_PATH / x)
)

# VisA annotation files don't directly provide our train/test column.
# We will leave this as "all" for now.
visa_metadata["split"] = "all"

# Add dataset name
visa_metadata["dataset"] = "VisA"

# Keep only the standardized columns
visa_metadata = visa_metadata[
    [
        "dataset",
        "category",
        "defect_type",
        "label",
        "image_path",
        "mask_path",
        "split"
    ]
]

print("Total metadata rows:", len(visa_metadata))
print("\nColumns:")
print(visa_metadata.columns.tolist())

visa_metadata.head()

Total metadata rows: 10821

Columns:
['dataset', 'category', 'defect_type', 'label', 'image_path', 'mask_path', 'split']


,dataset,category,defect_type,label,image_path,mask_path,split
0,VisA,candle,normal,0,c:\Users\rimjh\IndustrialDefectProject\dataset...,,all
1,VisA,candle,normal,0,c:\Users\rimjh\IndustrialDefectProject\dataset...,,all
2,VisA,candle,normal,0,c:\Users\rimjh\IndustrialDefectProject\dataset...,,all
3,VisA,candle,normal,0,c:\Users\rimjh\IndustrialDefectProject\dataset...,,all
4,VisA,candle,normal,0,c:\Users\rimjh\IndustrialDefectProject\dataset...,,all


In [8]:
# Inspect the VisA 1-class split file

split_file = VISA_PATH / "split_csv" / "1cls.csv"

visa_split = pd.read_csv(split_file)

print("Number of rows:", len(visa_split))

print("\nColumns:")
print(visa_split.columns.tolist())

print("\nFirst 10 rows:")
display(visa_split.head(10))

Number of rows: 10821

Columns:
['object', 'split', 'label', 'image', 'mask']

First 10 rows:


,object,split,label,image,mask
0,candle,train,normal,candle/Data/Images/Normal/0836.JPG,NaN
1,candle,train,normal,candle/Data/Images/Normal/0451.JPG,NaN
2,candle,train,normal,candle/Data/Images/Normal/0516.JPG,NaN
3,candle,train,normal,candle/Data/Images/Normal/0977.JPG,NaN
4,candle,train,normal,candle/Data/Images/Normal/0249.JPG,NaN
5,candle,train,normal,candle/Data/Images/Normal/0548.JPG,NaN
6,candle,train,normal,candle/Data/Images/Normal/0362.JPG,NaN
7,candle,train,normal,candle/Data/Images/Normal/0074.JPG,NaN
8,candle,train,normal,candle/Data/Images/Normal/0650.JPG,NaN
9,candle,train,normal,candle/Data/Images/Normal/0898.JPG,NaN


In [9]:
# Analyze the official VisA 1-class split

print("Split distribution:")
print(visa_split["split"].value_counts())

print("\nSplit + label distribution:")
print(
    visa_split
    .groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
)

Split distribution:
split
train    8659
test     2162
Name: count, dtype: int64

Split + label distribution:
label  anomaly  normal
split                 
test      1200     962
train        0    8659


In [10]:
# Add official VisA train/test split to our metadata

# Create a lookup table from the official split file
split_lookup = visa_split[
    ["image", "split"]
].copy()

# Normalize the image paths
split_lookup["image"] = split_lookup["image"].str.replace(
    "\\", "/", regex=False
)

visa_metadata["relative_image"] = visa_metadata["image_path"].apply(
    lambda x: str(Path(x).relative_to(VISA_PATH)).replace("\\", "/")
)

# Merge official split information
visa_metadata = visa_metadata.drop(columns=["split"])

visa_metadata = visa_metadata.merge(
    split_lookup,
    left_on="relative_image",
    right_on="image",
    how="left"
)

# Rename split column
visa_metadata = visa_metadata.rename(
    columns={"split": "split"}
)

# Remove temporary columns
visa_metadata = visa_metadata.drop(
    columns=["relative_image", "image"]
)

print("Split distribution:")
print(visa_metadata["split"].value_counts())

print("\nSplit + label distribution:")
print(
    visa_metadata
    .groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
)

Split distribution:
split
train    8659
test     2162
Name: count, dtype: int64

Split + label distribution:
label     0     1
split            
test    962  1200
train  8659     0


In [11]:
# Validate that every image has a split

print("Total metadata rows:", len(visa_metadata))
print("Missing splits:", visa_metadata["split"].isna().sum())

print("\nLabel distribution:")
print(visa_metadata["label"].value_counts())

print("\nMissing image paths:", 
      (~visa_metadata["image_path"].apply(lambda x: Path(x).exists())).sum())

print("\nMissing masks for anomalies:")
anomaly_metadata = visa_metadata[visa_metadata["label"] == 1]

print(
    (~anomaly_metadata["mask_path"].apply(lambda x: Path(x).exists())).sum()
)

Total metadata rows: 10821
Missing splits: 0

Label distribution:
label
0    9621
1    1200
Name: count, dtype: int64

Missing image paths: 0

Missing masks for anomalies:
0


In [13]:
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

# Metadata output folder
METADATA_PATH = PROJECT_ROOT / "metadata"
METADATA_PATH.mkdir(exist_ok=True)

print("Metadata folder:", METADATA_PATH)

Metadata folder: c:\Users\rimjh\IndustrialDefectProject\metadata


In [14]:
# Save VisA metadata

output_file = METADATA_PATH / "visa_metadata.csv"

visa_metadata.to_csv(output_file, index=False)

print("VisA metadata saved successfully!")
print("File:", output_file)
print("Rows:", len(visa_metadata))

VisA metadata saved successfully!
File: c:\Users\rimjh\IndustrialDefectProject\metadata\visa_metadata.csv
Rows: 10821
